In [0]:
import requests
import json
from pyspark.sql.functions import *
from pyspark.sql.types import *



In [0]:
spark.sql("CREATE CATALOG IF NOT EXISTS workspace")
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.default")
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.cricket_api_project")

base_path='/Volumes/workspace/default/cricket_api_project'

In [0]:
API_KEY = '1d6486d2-9565-4e5e-a66b-d2b2a6d75277'

api_url = f'https://api.cricapi.com/v1/countries?apikey=1d6486d2-9565-4e5e-a66b-d2b2a6d75277&offset=0'

response = requests.get(api_url)

response.raise_for_status()

api_data=response.json()
print(api_data.keys())
print(json.dumps(api_data,indent=2)[:2000])

In [0]:
raw_file_path = f'{base_path}/current_matches.json'

with open(raw_file_path,'w') as file:
    json.dump(api_data,file)

    print("Raw API saved:",raw_file_path)

In [0]:
bronze_data = [{
             "source_api":api_url,
             "raw_json":json.dumps(api_data),
             "ingestion_time":None

}]

bronze_schema=StructType([
    StructField("source-api",StringType(),True),
    StructField("raw_json",StringType(),True),
    StructField("ingestion_time",TimestampType(),True),
])

bronze_df = spark.createDataFrame(bronze_data,bronze_schema)\
            .withColumn("IngestionTime",current_timestamp())

display(bronze_df)


In [0]:
bronze_data = [{
             "source_api":api_url,
             "raw_json":json.dumps(api_data),
           
             }]

In [0]:
bronze_df = spark.createDataFrame(bronze_data)

In [0]:
from pyspark.sql.functions import current_timestamp

bronze_df = bronze_df.withColumn(
    "ingestion_time",
    current_timestamp()
)

display(bronze_df)

In [0]:
bronze_df.write\
     .format('delta')\
     .mode("overwrite")\
     .saveAsTable("workspace_default_current_matches") 

print("Bronze Table Created Successfully")   

In [0]:
%sql
select * from  workspace_default_current_matches

In [0]:
%sql
DROP TABLE IF EXISTS workspace.default.current_matches;

In [0]:
%sql
SHOW TABLES;

In [0]:
bronze_df.write\
    .format("delta")\
    .mode("overwrite")\
    .saveAsTable(" workspace.default.current_matches")

print("Bronze Successfully Created")    

In [0]:
%sql
select * from workspace.default.current_matches

In [0]:
%sql
show tables;